In [1]:
import os
import pandas as pd
from maomao.parsing.parsing_utils import *
from maomao.utils.constants import *

#### Processing and standardizing peptide datasets (PLPTP)

This notebook curates the **PLPTP** dataset by merging multiple provided test partitions, validating duplicate consistency, generating metadata, and exporting a clean standardized table for downstream modeling.

 - **Toxic effect / endpoint:** toxic
- **Source:** PLPTP
- **Sequence scope:** only non-modified peptide sequences are retained for the final dataset.

The pipeline performs the following steps:

- **Loads raw PLPTP files** from two folders (`Test1/` and `Test2/`), where each file is expected to contain:
  - `label` (first column)
  - `sequence` (second column)
- **Concatenates all files** into a single DataFrame with the standard schema:
  - `sequence`, `label`
- **Checks duplicated sequences**:
  - identical sequences with consistent labels are collapsed,
  - sequences appearing with conflicting labels are flagged as errors.
- **Builds metadata** from the project-wide Excel description sheet and adds dataset-level QC counters.
- **Exports curated outputs**:
  - `processed_toxic_dataset.csv`
  - `metadata.json`

In [2]:
name_source = "PLPTP"
name_task = "toxic_effect_classification"

# PATH_INPUT and PATH_EXPORT are imported from maomao.utils.constants
# Update them in constants.py according to the required input and export paths.

- Reading raw data

In [3]:
df_PLPTP = pd.concat([
    pd.read_csv(os.path.join(folder, file), header=None, names=["label", "sequence"])[["sequence", "label"]]
    for folder in [f"{PATH_INPUT}/{name_source}/Test1", 
                   f"{PATH_INPUT}/{name_source}/Test2"] 
    for file in os.listdir(folder)])
df_PLPTP.shape

(15026, 2)

- Checking duplicates

In [4]:
df_remove_duplicated, df_errors, df_unique = processing_duplicated(df_PLPTP, group_seq="sequence", sort_key="label")
df_full = pd.concat([df_unique, df_remove_duplicated], axis=0)

In [5]:
df_full.shape

(7513, 2)

In [6]:
df_errors.shape

(0, 1)

- Working with metada

In [7]:
df_metada = read_metadata("../../raw_data/raw_data_description.xlsx", name_source)
dict_metadata = create_metada_with_multiple_values(df_metada)

In [8]:
dict_metadata.update({
    "number_of_raw_sequences": int(len(df_PLPTP)),
    "number_of_sequences_retained": len(df_full),
    "number_of_positive_sequences": int((df_full["label"] == 1).sum()),
    "number_of_negative_sequences": int((df_full["label"] == 0).sum()),
    "number_of_erroneous_sequences": int(len(df_errors)),
    "modified_sequences_included": False,
})

dict_metadata

{'type source': 'Dataset',
 'static-dynamic': 'Static',
 'license': 'No information',
 'year of publication': 2025,
 'last update date': datetime.datetime(2025, 2, 23, 0, 0),
 'download date': Timestamp('2025-03-25 00:00:00'),
 'file format': 'txt;csv',
 'peptide property': 'toxic',
 'dataset information': 'Positive, Negative',
 'unit of measurement': 'No information',
 'obtaining negative dataset': 'Sampling from uniprot',
 'repository or server': 'https://github.com/birdsmart/PLPTP',
 'publication': 'https://www.sciencedirect.com/science/article/abs/pii/S0022283625001810',
 'number_of_raw_sequences': 15026,
 'number_of_sequences_retained': 7513,
 'number_of_positive_sequences': 2138,
 'number_of_negative_sequences': 5375,
 'number_of_erroneous_sequences': 0,
 'modified_sequences_included': False}

- Exporting data

In [9]:
os.makedirs(f"{PATH_EXPORT}/{name_task}/{name_source}/", exist_ok=True)
export_json(f"{PATH_EXPORT}/{name_task}/{name_source}/metadata.json", dict_metadata)

In [10]:
df_full.to_csv(f"{PATH_EXPORT}/{name_task}/{name_source}/processed_toxic_dataset.csv", index=False)